In [1]:
import pandas as pd
df = pd.read_parquet('/kaggle/input/notebooks/ajax0564/vyom-ai-gflinear2-final-data/multiclass_dataset.parquet')

df['is_multilabel'] = False
df.head(1)

,text,label,all_label,is_multilabel
0,... a delicious crime drama on par with the sl...,[very positive],"[very positive, positive, neutral, negative, v...",False


In [2]:
df1 = pd.read_parquet('/kaggle/input/notebooks/ajax0564/vyom-ai-gflinear2-final-data/multilabel_dataset.parquet')
df1 = df1.rename(columns={'labels':'label','all_labels':'all_label'})
df1['is_multilabel'] = True
df1.head()

,text,label,all_label,is_multilabel
0,Standard cosmological models fail to account f...,"[Energy Science, Astrophysics, Materials Scien...","[Genomics, Astrophysics, Materials Science, Ne...",True
1,Rising costs in advanced manufacturing are for...,"[Robotics, Electoral Politics & Campaigns]","[Artificial Intelligence, Cloud Computing, Cyb...",True
2,"I don't do the things you mentioned above, I j...","[disapproval, gratitude]","[admiration, amusement, anger, annoyance, appr...",True
3,Configure a comprehensive investigation into t...,"[Professional League Governance, Financial Tec...","[Artificial Intelligence, Cloud Computing, Cyb...",True
4,"Under cryogenic pressures exceeding 200 GPa, d...","[Astrophysics, Climate Science, Nuclear Physics]","[Genomics, Astrophysics, Materials Science, Ne...",True


In [3]:
df_all = pd.concat([df, df1], ignore_index=True)
df_all = df_all.sample(frac = 1)
df_all.shape

(157576, 4)

In [4]:
df_all.iloc[0]['is_multilabel']

np.False_

In [ ]:
set1 = df_all.sample(frac=0.9, random_state=42)

remaining_df = df_all.drop(set1.index)
set2 = remaining_df.sample(frac=0.1, random_state=42)

set1.to_parquet('multilabel_dataset_train.parquet',index=False)

set2.to_parquet('multilabel_dataset_val.parquet',index=False)

In [ ]:


import math
import random
from dataclasses import dataclass, fields
from typing import Any, Dict, List, Optional, Union

import pyarrow.parquet as pq
import torch
import torch.nn as nn
from accelerate import Accelerator, DataLoaderConfiguration, PartialState, DistributedDataParallelKwargs
from accelerate.utils import set_seed
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# Configure how data loaders are handled
dataloader_config = DataLoaderConfiguration(dispatch_batches=False)

random.seed(42)
torch.manual_seed(42)


@dataclass(frozen=True)
class SpecialTokens:
    SEP_STRUCT: str = "[SEP_STRUCT]"
    SEP_TEXT: str = "[SEP_TEXT]"
    P_TOKEN: str = "[P]"
    C_TOKEN: str = "[C]"
    E_TOKEN: str = "[E]"
    R_TOKEN: str = "[R]"
    L_TOKEN: str = "[L]"
    EXAMPLE_TOKEN: str = "[EXAMPLE]"
    OUTPUT_TOKEN: str = "[OUTPUT]"
    DESC_TOKEN: str = "[DESCRIPTION]"

    @property
    def SPECIAL_TOKENS(self) -> List[str]:
        """Returns all string field values as a list."""
        return [getattr(self, f.name) for f in fields(self)]


def sample_labels(
    all_candidate_labels: List[str],
    correct_labels: List[str],
    max_negatives: int = 10,
) -> List[str]:
    """Subsamples negative labels while retaining ground-truth positive labels with strict deduplication."""
    correct_set = list(dict.fromkeys(correct_labels))
    negative_candidates = [
        label for label in dict.fromkeys(all_candidate_labels) if label not in set(correct_set)
    ]

    num_negatives = min(len(negative_candidates), max_negatives)
    sampled_negatives = random.sample(negative_candidates, num_negatives)

    combined_labels = correct_set + sampled_negatives
    random.shuffle(combined_labels)
    return combined_labels


def format_input(
    text: str,
    candidate_labels: List[str],
    special_tokens: Any,
    is_multilabel: bool,
) -> str:
    task_type = "topics:" if is_multilabel else "sentiment:"
    label_prefix = " ".join(
        [f"{special_tokens.L_TOKEN} {label}" for label in candidate_labels]
    )
    
    prefix = f"{special_tokens.P_TOKEN} {task_type} {label_prefix}"
    full_text = f"{prefix} {special_tokens.SEP_TEXT} {text}"
    return full_text


class ParquetIterableClassificationDataset(IterableDataset):
    def __init__(
        self,
        parquet_path: str,
        tokenizer: Any,
        special_tokens: Any,
        max_negatives: Optional[int] = 10,
        batch_size: int = 64,
    ):
        self.parquet_path = parquet_path
        self.tokenizer = tokenizer
        self.special_tokens = special_tokens
        self.max_negatives = max_negatives
        self.batch_size = batch_size
        
        # Cache row count 
        pf = pq.ParquetFile(self.parquet_path)
        self._num_rows = pf.metadata.num_rows

    def __len__(self) -> int:
        return self._num_rows

    def __iter__(self):
        pf = pq.ParquetFile(self.parquet_path)
        num_row_groups = pf.num_row_groups

        # Shard across GPUs / DDP processes
        state = PartialState()
        per_process = int(math.ceil(num_row_groups / float(state.num_processes)))
        process_start = state.process_index * per_process
        process_end = min(process_start + per_process, num_row_groups)
        process_row_groups = list(range(process_start, process_end))

        # Shard across DataLoader workers (if num_workers > 0)
        worker_info = get_worker_info()
        if worker_info is None:
            row_groups = process_row_groups
        else:
            num_process_groups = len(process_row_groups)
            per_worker = int(
                math.ceil(num_process_groups / float(worker_info.num_workers))
            )
            worker_id = worker_info.id
            worker_start = worker_id * per_worker
            worker_end = min(worker_start + per_worker, num_process_groups)
            row_groups = process_row_groups[worker_start:worker_end]

        l_token_id = self.tokenizer.convert_tokens_to_ids(
            self.special_tokens.L_TOKEN
        )

        for rg_idx in row_groups:
            row_group = pf.read_row_group(rg_idx)
            for batch in row_group.to_batches(max_chunksize=self.batch_size):
                pydict = batch.to_pydict()
                texts = pydict["text"]
                labels_list = pydict["all_label"]
                correct_labels = pydict["label"]
                is_multilabel_col = pydict["is_multilabel"]

                for text, all_labels, ground_truth, is_multi in zip(
                    texts, labels_list, correct_labels, is_multilabel_col
                ):
                    ground_truth = list(dict.fromkeys(ground_truth))
                    all_labels = list(dict.fromkeys(all_labels))

                    if is_multi and self.max_negatives is not None:
                        candidate_labels = sample_labels(
                            all_candidate_labels=all_labels,
                            correct_labels=ground_truth,
                            max_negatives=self.max_negatives,
                        )
                    else:
                        candidate_labels = list(all_labels)
                        random.shuffle(candidate_labels)

                    full_text = format_input(
                        text=text,
                        candidate_labels=candidate_labels,
                        special_tokens=self.special_tokens,
                        is_multilabel=is_multi,
                    )

                    encoding = self.tokenizer(
                        full_text,
                        truncation=True,
                        return_tensors=None,
                        max_length=384,
                    )
                    input_ids = encoding["input_ids"]

                    targets = [
                        1.0 if label in ground_truth else 0.0
                        for label in candidate_labels
                    ]

                    # Match targets to actual surviving [L] tokens after truncation
                    num_label_tokens = input_ids.count(l_token_id)
                    
                    # Guard against zero surviving label tokens
                    if num_label_tokens == 0:
                        continue
                        
                    targets = targets[:num_label_tokens]

                    yield {
                        "input_ids": torch.tensor(
                            input_ids, dtype=torch.long
                        ),
                        "is_multilabel": is_multi,
                        "targets": targets,
                    }


def collate_fn(batch, pad_token_id=0):
    input_ids_list = []
    attention_mask_list = []
    flat_targets = []
    is_multilabel_list = []
    num_labels_per_sample = []

    for item in batch:
        encoding = item["input_ids"]
        if isinstance(encoding, dict):
            input_ids = encoding["input_ids"]
            mask = encoding.get(
                "attention_mask", torch.ones_like(input_ids)
            )
        else:
            input_ids = encoding
            mask = torch.ones_like(input_ids)

        if not isinstance(input_ids, torch.Tensor):
            input_ids = torch.tensor(input_ids, dtype=torch.long)
        if not isinstance(mask, torch.Tensor):
            mask = torch.tensor(mask, dtype=torch.long)

        input_ids_list.append(input_ids)
        attention_mask_list.append(mask)

        flat_targets.extend(item["targets"])
        is_multilabel_list.append(item["is_multilabel"])
        num_labels_per_sample.append(len(item["targets"]))

    padded_input_ids = pad_sequence(
        input_ids_list, batch_first=True, padding_value=pad_token_id
    )
    padded_attention_mask = pad_sequence(
        attention_mask_list, batch_first=True, padding_value=0
    )

    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "targets": torch.tensor(flat_targets, dtype=torch.float32),
        "is_multilabel": torch.tensor(is_multilabel_list, dtype=torch.bool),
        "num_labels_per_sample": num_labels_per_sample,
    }


class GLiNERTextClassifier(nn.Module):
    def __init__(self, base_model: nn.Module, label_token_id: int):
        super().__init__()
        self.encoder = base_model
        self.hidden_size = self.encoder.config.hidden_size
        self.label_token_id = label_token_id

        self.label_ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.hidden_size // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        )
        sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Extract label representation at [L] token positions
        label_mask = input_ids == self.label_token_id
        label_features = sequence_output[label_mask]  # [total_labels_in_batch, hidden_size]

        refined_labels = self.label_ffn(label_features)

        logits = self.classifier(refined_labels).squeeze(-1)
        return logits


class HybridClassificationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.ce = nn.CrossEntropyLoss()

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        num_labels_per_sample: List[int],
        is_multilabel: torch.Tensor,
    ) -> torch.Tensor:
        offset = 0
        sample_losses = []

        for b, is_multi in enumerate(is_multilabel):
            n_labels = num_labels_per_sample[b]
            if n_labels == 0:
                continue

            sample_logits = logits[offset : offset + n_labels]
            sample_targets = targets[offset : offset + n_labels]

            if is_multi:
                loss = self.bce(sample_logits, sample_targets)
            else:
                # FIX: Check if positive ground truth exists before calling CrossEntropy
                has_positive = (sample_targets == 1.0).any()
                if has_positive:
                    target_idx = torch.argmax(sample_targets)
                    loss = self.ce(sample_logits.unsqueeze(0), target_idx.unsqueeze(0))
                else:
                    # Fallback to BCE if truncation dropped the positive candidate
                    loss = self.bce(sample_logits, sample_targets)

            sample_losses.append(loss)
            offset += n_labels

        if not sample_losses:
            return logits.sum() * 0.0

        return torch.stack(sample_losses).mean()
        
@torch.no_grad()
def evaluate(
    model: nn.Module,
    val_dataloader: DataLoader,
    accelerator: Accelerator,
    tokenizer: Any,
    special_tokens: Any,
    criterion: nn.Module,
    threshold: float = 0.5,
    num_samples_to_print: int = 10,
) -> Dict[str, float]:
    model.eval()

    total_loss = torch.tensor(0.0, device=accelerator.device)
    total_samples = torch.tensor(0.0, device=accelerator.device)

    tp = torch.tensor(0.0, device=accelerator.device)
    fp = torch.tensor(0.0, device=accelerator.device)
    fn = torch.tensor(0.0, device=accelerator.device)
    tn = torch.tensor(0.0, device=accelerator.device)

    sample_predictions = []
    special_ids_set = set(tokenizer.all_special_ids)
    label_token_id = tokenizer.convert_tokens_to_ids(special_tokens.L_TOKEN)

    for batch in val_dataloader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        targets = batch["targets"]
        num_labels_per_sample = batch["num_labels_per_sample"]
        is_multilabel_tensor = batch["is_multilabel"]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        loss = criterion(
            logits=logits,
            targets=targets,
            num_labels_per_sample=num_labels_per_sample,
            is_multilabel=is_multilabel_tensor,
        )

        batch_num_samples = len(is_multilabel_tensor)
        total_loss += loss.detach() * batch_num_samples
        total_samples += batch_num_samples

        is_multilabel_list = (
            is_multilabel_tensor.tolist()
            if isinstance(is_multilabel_tensor, torch.Tensor)
            else is_multilabel_tensor
        )

        curr_offset = 0

        for b in range(input_ids.size(0)):
            n_labels = num_labels_per_sample[b]
            if n_labels == 0:
                continue

            is_multi = is_multilabel_list[b]
            sample_logits = logits[curr_offset : curr_offset + n_labels]
            sample_targets = targets[curr_offset : curr_offset + n_labels]

            # 1. FIX: Task-specific prediction logic
            if is_multi:
                # Multi-label: Independent thresholding
                sample_probs = torch.sigmoid(sample_logits)
                sample_preds = (sample_probs >= threshold).float()
            else:
                # Multi-class: Relative Argmax pick
                sample_probs = (
                    torch.softmax(sample_logits, dim=-1)
                    if sample_logits.numel() > 1
                    else torch.sigmoid(sample_logits)
                )
                sample_preds = torch.zeros_like(sample_logits)
                if sample_logits.numel() > 0:
                    pred_idx = torch.argmax(sample_logits)
                    sample_preds[pred_idx] = 1.0

            # 2. Accumulate confusion matrix values per sample
            tp += (sample_preds * sample_targets).sum()
            fp += (sample_preds * (1.0 - sample_targets)).sum()
            fn += ((1.0 - sample_preds) * sample_targets).sum()
            tn += ((1.0 - sample_preds) * (1.0 - sample_targets)).sum()

            # 3. Format visual sample logs for printing
            if (
                len(sample_predictions) < num_samples_to_print
                and accelerator.is_main_process
            ):
                ids = input_ids[b].tolist()
                full_text = tokenizer.decode(ids, skip_special_tokens=False)

                if special_tokens.SEP_TEXT in full_text:
                    text = (
                        full_text.split(special_tokens.SEP_TEXT)[-1]
                        .replace(tokenizer.pad_token, "")
                        .strip()
                    )
                else:
                    text = full_text.replace(tokenizer.pad_token, "").strip()

                labels = []
                for i, token_id in enumerate(ids):
                    if token_id == label_token_id:
                        label_tokens = []
                        for j in range(i + 1, len(ids)):
                            if ids[j] in special_ids_set:
                                break
                            label_tokens.append(ids[j])
                        decoded_label = tokenizer.decode(label_tokens).strip()
                        labels.append(decoded_label if decoded_label else "[UNK]")

                labels = labels[:n_labels]
                sample_targets_list = sample_targets.tolist()
                sample_probs_list = sample_probs.tolist()
                sample_preds_list = sample_preds.tolist()

                ground_truth_labels = [
                    lbl
                    for lbl, tgt in zip(labels, sample_targets_list)
                    if tgt == 1.0
                ]

                predicted_labels = []
                confidences = []

                for lbl, prob, pred in zip(
                    labels, sample_probs_list, sample_preds_list
                ):
                    if pred == 1.0:
                        predicted_labels.append(lbl)
                        confidences.append(round(prob, 4))

                sample_predictions.append(
                    {
                        "type": "Multi-Label" if is_multi else "Multi-Class",
                        "text": text,
                        "candidate_labels": labels,
                        "ground_truth_labels": ground_truth_labels,
                        "predicted_labels": predicted_labels,
                        "confidence": confidences,
                    }
                )

            curr_offset += n_labels

    # Gather metrics across GPU processes
    total_loss = accelerator.reduce(total_loss, reduction="sum").item()
    total_samples = accelerator.reduce(total_samples, reduction="sum").item()

    tp = accelerator.reduce(tp, reduction="sum").item()
    fp = accelerator.reduce(fp, reduction="sum").item()
    fn = accelerator.reduce(fn, reduction="sum").item()
    tn = accelerator.reduce(tn, reduction="sum").item()

    # Print representative predictions
    if accelerator.is_main_process:
        accelerator.print("\n" + "=" * 30 + " EVALUATION SAMPLES " + "=" * 30)
        for idx, sample in enumerate(sample_predictions, 1):
            accelerator.print(f"\nSample {idx} [{sample['type']}]:")
            accelerator.print(f"  Text             : {sample['text']}")
            accelerator.print(f"  Candidates       : {sample['candidate_labels']}")
            accelerator.print(f"  Ground Truth     : {sample['ground_truth_labels']}")
            accelerator.print(f"  Predicted        : {sample['predicted_labels']}")
            accelerator.print(f"  Confidence       : {sample['confidence']}")
        accelerator.print("=" * 80 + "\n")

    avg_loss = total_loss / max(total_samples, 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

    return {
        "val_loss": round(avg_loss, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4),
    }

def main():
    set_seed(42)

    use_fp16 = "fp16" if torch.cuda.is_available() else "no"
    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)

    # FIX: Pass ddp_kwargs handler into Accelerator
    accelerator = Accelerator(
        mixed_precision=use_fp16,
        dataloader_config=dataloader_config,
        kwargs_handlers=[ddp_kwargs],
    )
    special_tokens = SpecialTokens()

    model_name = "FacebookAI/roberta-base"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name, add_pooling_layer=False)

    num_added_tokens = tokenizer.add_special_tokens(
        {"additional_special_tokens": special_tokens.SPECIAL_TOKENS}
    )

    if num_added_tokens > 0:
        base_model.resize_token_embeddings(len(tokenizer))

    label_token_id = tokenizer.convert_tokens_to_ids(special_tokens.L_TOKEN)
    model = GLiNERTextClassifier(
        base_model=base_model, label_token_id=label_token_id
    )
    if accelerator.is_main_process:
        print(f"Added {num_added_tokens} new tokens to tokenizer.")

    train_data = ParquetIterableClassificationDataset(
        "/kaggle/working/multilabel_dataset_train.parquet",
        tokenizer=tokenizer,
        special_tokens=special_tokens,
    )

    train_loader = DataLoader(
        train_data,
        batch_size=48,
        collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
    )

    val_data = ParquetIterableClassificationDataset(
        "/kaggle/working/multilabel_dataset_val.parquet",
        tokenizer=tokenizer,
        special_tokens=special_tokens,
        max_negatives=15,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=64,
        collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = HybridClassificationLoss()

    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )

    num_epochs = 2
    total_rows = len(train_data)
    per_process_rows = math.ceil(total_rows / accelerator.num_processes)
    total_train_batches = math.ceil(per_process_rows / 48)

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        progress_bar = tqdm(
            train_loader,
            total=total_train_batches,
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            disable=not accelerator.is_local_main_process,
        )

        for step, batch in enumerate(progress_bar):
            optimizer.zero_grad()

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            loss = criterion(
                logits=logits,
                targets=batch["targets"],
                num_labels_per_sample=batch["num_labels_per_sample"],
                is_multilabel=batch["is_multilabel"],
            )

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()
            current_avg_loss = total_train_loss / (step + 1)
            progress_bar.set_postfix({"loss": f"{current_avg_loss:.4f}"})

        val_metrics = evaluate(
            model=model,
            val_dataloader=val_loader,
            accelerator=accelerator,
            special_tokens=special_tokens,
            tokenizer=tokenizer,
            threshold=0.5,
            criterion=criterion,
            num_samples_to_print=20,
        )
        accelerator.print(f"Epoch {epoch + 1} Validation Metrics: {val_metrics}")

    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    if accelerator.is_main_process:
        torch.save(unwrapped_model.state_dict(), "gliner_classifier.pt")
        tokenizer.save_pretrained("./tokenizer")
        accelerator.print("Model successfully saved!")


if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Added 10 new tokens to tokenizer.


Epoch 1/2:   0%|          | 0/2955 [00:00<?, ?it/s]


============================== EVALUATION SAMPLES ==============================

Sample 1 [Multi-Class]:
  Text             : Karzai set to be Afghan president in ring of steel They are two of the most hawkish members of President George W. Bush #39;s cabinet and key architects of. the Washington-backed war that overthrew the Taliban in the wake of the September 11, 2001 attacks.</s>
  Candidates       : ['Sports', 'World', 'Business', 'Sci/Tech']
  Ground Truth     : ['World']
  Predicted        : ['World']
  Confidence       : [0.9967]

Sample 2 [Multi-Label]:
  Text             : Polymer matrices doped with photoreactive organic dyes exhibit enhanced stress resilience under extreme thermal gradients, a property being leveraged in next-generation aerospace composites. These materials, synthesized via controlled radical polymerization, demonstrate tunable chromophore distributions that allow real-time optical monitoring of structural integrity during flight conditions. In hypersonic

Epoch 2/2:   0%|          | 0/2955 [00:00<?, ?it/s]


============================== EVALUATION SAMPLES ==============================

Sample 1 [Multi-Class]:
  Text             : Karzai set to be Afghan president in ring of steel They are two of the most hawkish members of President George W. Bush #39;s cabinet and key architects of. the Washington-backed war that overthrew the Taliban in the wake of the September 11, 2001 attacks.</s>
  Candidates       : ['World', 'Sci/Tech', 'Business', 'Sports']
  Ground Truth     : ['World']
  Predicted        : ['World']
  Confidence       : [0.9989]

Sample 2 [Multi-Label]:
  Text             : Polymer matrices doped with photoreactive organic dyes exhibit enhanced stress resilience under extreme thermal gradients, a property being leveraged in next-generation aerospace composites. These materials, synthesized via controlled radical polymerization, demonstrate tunable chromophore distributions that allow real-time optical monitoring of structural integrity during flight conditions. In hypersonic

In [7]:
# # %%writefile train_script.py

# import math
# import random
# from dataclasses import dataclass, fields
# from typing import Any, Dict, List, Optional, Union

# import pyarrow.parquet as pq
# import torch
# import torch.nn as nn
# from accelerate import Accelerator, DataLoaderConfiguration, PartialState,DistributedDataParallelKwargs
# from accelerate.utils import set_seed
# from torch.nn.utils.rnn import pad_sequence
# from torch.utils.data import DataLoader, IterableDataset, get_worker_info
# from tqdm.auto import tqdm
# from transformers import AutoModel, AutoTokenizer

# # Configure how data loaders are handled
# dataloader_config = DataLoaderConfiguration(dispatch_batches=False)

# random.seed(42)
# torch.manual_seed(42)


# @dataclass(frozen=True)
# class SpecialTokens:
#     SEP_STRUCT: str = "[SEP_STRUCT]"
#     SEP_TEXT: str = "[SEP_TEXT]"
#     P_TOKEN: str = "[P]"
#     C_TOKEN: str = "[C]"
#     E_TOKEN: str = "[E]"
#     R_TOKEN: str = "[R]"
#     L_TOKEN: str = "[L]"
#     EXAMPLE_TOKEN: str = "[EXAMPLE]"
#     OUTPUT_TOKEN: str = "[OUTPUT]"
#     DESC_TOKEN: str = "[DESCRIPTION]"

#     @property
#     def SPECIAL_TOKENS(self) -> List[str]:
#         """Returns all string field values as a list."""
#         return [getattr(self, f.name) for f in fields(self)]


# def sample_labels(
#     all_candidate_labels: List[str],
#     correct_labels: List[str],
#     max_negatives: int = 10,
# ) -> List[str]:
#     """Subsamples negative labels while retaining ground-truth positive labels with strict deduplication."""
#     correct_set = list(dict.fromkeys(correct_labels))
#     negative_candidates = [
#         label for label in dict.fromkeys(all_candidate_labels) if label not in set(correct_set)
#     ]

#     num_negatives = min(len(negative_candidates), max_negatives)
#     sampled_negatives = random.sample(negative_candidates, num_negatives)

#     combined_labels = correct_set + sampled_negatives
#     random.shuffle(combined_labels)
#     return combined_labels


# def format_input(
#     text: str,
#     candidate_labels: List[str],
#     special_tokens: Any,
#     is_multilabel: bool,
# ) -> str:
#     task_type = "topics:" if is_multilabel else "sentiment:"
#     label_prefix = " ".join(
#         [f"{special_tokens.L_TOKEN} {label}" for label in candidate_labels]
#     )
    
#     # Place task_type before candidate labels so the last label ends cleanly at SEP_TEXT
#     prefix = f"{special_tokens.P_TOKEN} {task_type} {label_prefix}"
#     full_text = f"{prefix} {special_tokens.SEP_TEXT} {text}"
#     return full_text


# class ParquetIterableClassificationDataset(IterableDataset):
#     def __init__(
#         self,
#         parquet_path: str,
#         tokenizer: Any,
#         special_tokens: Any,
#         max_negatives: Optional[int] = 10,
#         batch_size: int = 64,
#     ):
#         self.parquet_path = parquet_path
#         self.tokenizer = tokenizer
#         self.special_tokens = special_tokens
#         self.max_negatives = max_negatives
#         self.batch_size = batch_size

#     def __len__(self) -> int:
#         pf = pq.ParquetFile(self.parquet_path)
#         self._num_rows = pf.metadata.num_rows
#         return self._num_rows

#     def __iter__(self):
#         pf = pq.ParquetFile(self.parquet_path)
#         num_row_groups = pf.num_row_groups

#         # 1. Shard across GPUs / DDP processes
#         state = PartialState()
#         per_process = int(math.ceil(num_row_groups / float(state.num_processes)))
#         process_start = state.process_index * per_process
#         process_end = min(process_start + per_process, num_row_groups)
#         process_row_groups = list(range(process_start, process_end))

#         # 2. Shard across DataLoader workers (if num_workers > 0)
#         worker_info = get_worker_info()
#         if worker_info is None:
#             row_groups = process_row_groups
#         else:
#             num_process_groups = len(process_row_groups)
#             per_worker = int(
#                 math.ceil(num_process_groups / float(worker_info.num_workers))
#             )
#             worker_id = worker_info.id
#             worker_start = worker_id * per_worker
#             worker_end = min(worker_start + per_worker, num_process_groups)
#             row_groups = process_row_groups[worker_start:worker_end]

#         l_token_id = self.tokenizer.convert_tokens_to_ids(
#             self.special_tokens.L_TOKEN
#         )

#         for rg_idx in row_groups:
#             row_group = pf.read_row_group(rg_idx)
#             for batch in row_group.to_batches(max_chunksize=self.batch_size):
#                 pydict = batch.to_pydict()
#                 texts = pydict["text"]
#                 labels_list = pydict["all_label"]
#                 correct_labels = pydict["label"]

#                 # Extract the dynamic is_multilabel column
#                 is_multilabel_col = pydict["is_multilabel"]

#                 # Iterate through all columns, including is_multi
#                 for text, all_labels, ground_truth, is_multi in zip(
#                     texts, labels_list, correct_labels, is_multilabel_col
#                 ):
#                     ground_truth = list(dict.fromkeys(ground_truth))
#                     all_labels = list(dict.fromkeys(all_labels))

#                     if is_multi:
#                         # Only apply negative sampling if max_negatives is not None
#                         if self.max_negatives is not None:
#                             candidate_labels = sample_labels(
#                                 all_candidate_labels=all_labels,
#                                 correct_labels=ground_truth,
#                                 max_negatives=self.max_negatives,
#                             )
#                         else:
#                             candidate_labels = list(all_labels)
#                             random.shuffle(candidate_labels)
#                     else:
#                         candidate_labels = list(all_labels)
#                         random.shuffle(candidate_labels)

#                     full_text = format_input(
#                         text=text,
#                         candidate_labels=candidate_labels,
#                         special_tokens=self.special_tokens,
#                         is_multilabel=is_multi,  # Pass the row's specific boolean
#                     )

#                     encoding = self.tokenizer(
#                         full_text,
#                         truncation=True,
#                         return_tensors=None,
#                         max_length=384,
#                     )
#                     input_ids = encoding["input_ids"]

#                     targets = [
#                         1.0 if label in ground_truth else 0.0
#                         for label in candidate_labels
#                     ]

#                     # Match targets to actual surviving [L] tokens after truncation
#                     num_label_tokens = input_ids.count(l_token_id)
#                     targets = targets[:num_label_tokens]

#                     yield {
#                         "input_ids": torch.tensor(
#                             input_ids, dtype=torch.long
#                         ),
#                         "is_multilabel": is_multi,  # Yield the row's specific boolean
#                         "targets": targets,
#                     }


# def collate_fn(batch, pad_token_id=0):
#     input_ids_list = []
#     attention_mask_list = []
#     flat_targets = []
#     is_multilabel_list = []
#     num_labels_per_sample = []

#     for item in batch:
#         encoding = item["input_ids"]
#         if isinstance(encoding, dict):
#             input_ids = encoding["input_ids"]
#             mask = encoding.get(
#                 "attention_mask", torch.ones_like(input_ids)
#             )
#         else:
#             input_ids = encoding
#             mask = torch.ones_like(input_ids)

#         if not isinstance(input_ids, torch.Tensor):
#             input_ids = torch.tensor(input_ids, dtype=torch.long)
#         if not isinstance(mask, torch.Tensor):
#             mask = torch.tensor(mask, dtype=torch.long)

#         input_ids_list.append(input_ids)
#         attention_mask_list.append(mask)

#         # Append targets directly to flat 1D list
#         flat_targets.extend(item["targets"])
#         is_multilabel_list.append(item["is_multilabel"])
#         num_labels_per_sample.append(len(item["targets"]))

#     padded_input_ids = pad_sequence(
#         input_ids_list, batch_first=True, padding_value=pad_token_id
#     )
#     padded_attention_mask = pad_sequence(
#         attention_mask_list, batch_first=True, padding_value=0
#     )

#     return {
#         "input_ids": padded_input_ids,
#         "attention_mask": padded_attention_mask,
#         # 1D Target Tensor: Shape [total_label_tokens_in_batch]
#         "targets": torch.tensor(flat_targets, dtype=torch.float32),
#         "is_multilabel": torch.tensor(is_multilabel_list, dtype=torch.bool),
#         "num_labels_per_sample": num_labels_per_sample,
#     }


# @torch.no_grad()
# def evaluate(
#     model: nn.Module,
#     val_dataloader: DataLoader,
#     accelerator: Accelerator,
#     tokenizer: Any,
#     special_tokens: Any,
#     threshold: float = 0.5,
#     criterion: nn.Module = nn.BCEWithLogitsLoss(),
#     num_samples_to_print: int = 10,
# ) -> Dict[str, float]:
#     model.eval()

#     total_loss = torch.tensor(0.0, device=accelerator.device)
#     total_label_count = torch.tensor(0.0, device=accelerator.device)

#     tp = torch.tensor(0.0, device=accelerator.device)
#     fp = torch.tensor(0.0, device=accelerator.device)
#     fn = torch.tensor(0.0, device=accelerator.device)
#     tn = torch.tensor(0.0, device=accelerator.device)

#     sample_predictions = []
#     special_ids_set = set(tokenizer.all_special_ids)

#     for batch in val_dataloader:
#         input_ids = batch["input_ids"]
#         logits = model(
#             input_ids=input_ids, attention_mask=batch["attention_mask"]
#         )
#         targets = batch["targets"]
#         loss = criterion(logits=logits,targets=batch["targets"],num_labels_per_sample=batch["num_labels_per_sample"],is_multilabel=batch["is_multilabel"])

#         # loss = criterion(logits, targets)

#         num_labels_in_batch = targets.numel()
#         total_loss += loss * num_labels_in_batch
#         total_label_count += num_labels_in_batch

#         probs = torch.sigmoid(logits)
#         preds = (probs >= threshold).float()

#         tp += (preds * targets).sum()
#         fp += (preds * (1 - targets)).sum()
#         fn += ((1 - preds) * targets).sum()
#         tn += ((1 - preds) * (1 - targets)).sum()

#         if (
#             len(sample_predictions) < num_samples_to_print
#             and accelerator.is_main_process
#         ):
#             label_token_id = tokenizer.convert_tokens_to_ids(
#                 special_tokens.L_TOKEN
#             )
#             num_labels_per_sample = batch["num_labels_per_sample"]
#             is_multilabel_list = batch["is_multilabel"].tolist()

#             curr_offset = 0
#             for b in range(input_ids.size(0)):
#                 if len(sample_predictions) >= num_samples_to_print:
#                     break

#                 n_labels = num_labels_per_sample[b]
#                 is_multi = is_multilabel_list[b]

#                 sample_logits = logits[curr_offset : curr_offset + n_labels]
#                 sample_targets = targets[curr_offset : curr_offset + n_labels].tolist()

#                 # Always use Sigmoid because model was trained with BCEWithLogitsLoss
#                 sample_probs = torch.sigmoid(sample_logits).tolist()

#                 ids = input_ids[b].tolist()
#                 full_text = tokenizer.decode(ids, skip_special_tokens=False)
#                 if special_tokens.SEP_TEXT in full_text:
#                     text = (
#                         full_text.split(special_tokens.SEP_TEXT)[-1]
#                         .replace(tokenizer.pad_token, "")
#                         .strip()
#                     )
#                 else:
#                     text = full_text.replace(tokenizer.pad_token, "").strip()

#                 # FIX 1: Properly decode ALL subwords following [L] until the next special token
#                 labels = []
#                 for i, token_id in enumerate(ids):
#                     if token_id == label_token_id:
#                         label_tokens = []
#                         for j in range(i + 1, len(ids)):
#                             if ids[j] in special_ids_set:
#                                 break
#                             label_tokens.append(ids[j])
#                         decoded_label = tokenizer.decode(label_tokens).strip()
#                         if decoded_label:
#                             labels.append(decoded_label)

#                 # Ensure extracted label length aligns with logit count
#                 labels = labels[:n_labels]

#                 ground_truth_labels = []
#                 predicted_labels = []
#                 confidences = []

#                 if not is_multi:
#                     # Single-class: Select candidate with highest sigmoid probability
#                     max_idx = int(torch.argmax(sample_logits).item())
#                     if max_idx < len(labels):
#                         predicted_labels.append(labels[max_idx])
#                         confidences.append(round(sample_probs[max_idx], 4))
#                 else:
#                     # Multi-label: Select candidates with sigmoid prob >= threshold
#                     for label, prob in zip(labels, sample_probs):
#                         if prob >= threshold:
#                             predicted_labels.append(label)
#                             confidences.append(round(prob, 4))

#                 for label, target in zip(labels, sample_targets):
#                     if target == 1.0:
#                         ground_truth_labels.append(label)

#                 sample_predictions.append(
#                     {
#                         "type": "Multi-Label" if is_multi else "Multi-Class",
#                         "text": text,
#                         "candidate_labels": labels,
#                         "ground_truth_labels": ground_truth_labels,
#                         "predicted_labels": predicted_labels,
#                         "confidence": confidences,
#                     }
#                 )

#                 curr_offset += n_labels

#     total_loss = accelerator.reduce(total_loss, reduction="sum").item()
#     total_label_count = accelerator.reduce(total_label_count, reduction="sum").item()

#     tp = accelerator.reduce(tp, reduction="sum").item()
#     fp = accelerator.reduce(fp, reduction="sum").item()
#     fn = accelerator.reduce(fn, reduction="sum").item()
#     tn = accelerator.reduce(tn, reduction="sum").item()

#     if accelerator.is_main_process:
#         accelerator.print("\n" + "=" * 30 + " EVALUATION SAMPLES " + "=" * 30)
#         for idx, sample in enumerate(sample_predictions, 1):
#             accelerator.print(f"\nSample {idx} [{sample['type']}]:")
#             accelerator.print(f"  Text             : {sample['text']}")
#             accelerator.print(f"  Candidates       : {sample['candidate_labels']}")
#             accelerator.print(f"  Ground Truth     : {sample['ground_truth_labels']}")
#             accelerator.print(f"  Predicted        : {sample['predicted_labels']}")
#             accelerator.print(f"  Confidence       : {sample['confidence']}")
#         accelerator.print("=" * 80 + "\n")

#     avg_loss = total_loss / max(total_label_count, 1e-8)
#     precision = tp / (tp + fp + 1e-8)
#     recall = tp / (tp + fn + 1e-8)
#     f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
#     accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

#     return {
#         "val_loss": round(avg_loss, 4),
#         "precision": round(precision, 4),
#         "recall": round(recall, 4),
#         "f1": round(f1, 4),
#         "accuracy": round(accuracy, 4),
#     }
    
# class GLiNERTextClassifier(nn.Module):
#     def __init__(self, base_model: nn.Module, label_token_id: int):
#         super().__init__()
#         self.encoder = base_model
#         self.hidden_size = self.encoder.config.hidden_size
#         self.label_token_id = label_token_id

#         self.label_ffn = nn.Sequential(
#             nn.Linear(self.hidden_size, self.hidden_size),
#             nn.GELU(),
#             nn.Linear(self.hidden_size, self.hidden_size),
#         )

#         self.classifier = nn.Sequential(
#             nn.Linear(self.hidden_size, self.hidden_size // 2),
#             nn.GELU(),
#             nn.Linear(self.hidden_size // 2, 1),
#         )

#     def forward(self, input_ids, attention_mask):
#         outputs = self.encoder(
#             input_ids=input_ids, attention_mask=attention_mask
#         )
#         sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

#         # Extract label representation at [L] token positions directly into a 1D sequence
#         label_mask = input_ids == self.label_token_id
#         label_features = sequence_output[label_mask]  # [total_labels_in_batch, hidden_size]

#         refined_labels = self.label_ffn(label_features)

#         # Output 1D Tensor logits of shape [total_labels_in_batch]
#         logits = self.classifier(refined_labels).squeeze(-1)
#         return logits


# class HybridClassificationLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.bce = nn.BCEWithLogitsLoss()
#         self.ce = nn.CrossEntropyLoss()

#     def forward(
#         self,
#         logits: torch.Tensor,
#         targets: torch.Tensor,
#         num_labels_per_sample: List[int],
#         is_multilabel: torch.Tensor,
#     ) -> torch.Tensor:
#         """
#         Calculates Softmax CrossEntropy for single-label samples and 
#         BCEWithLogitsLoss for multi-label samples within the batch.
#         """
#         offset = 0
#         sample_losses = []

#         for b, is_multi in enumerate(is_multilabel):
#             n_labels = num_labels_per_sample[b]
#             sample_logits = logits[offset : offset + n_labels]
#             sample_targets = targets[offset : offset + n_labels]

#             if is_multi:
#                 # Multi-label: Independent binary classification across candidates
#                 loss = self.bce(sample_logits, sample_targets)
#             else:
#                 # Multi-class: Softmax across candidate logits
#                 target_idx = torch.argmax(sample_targets)
#                 loss = self.ce(sample_logits.unsqueeze(0), target_idx.unsqueeze(0))

#             sample_losses.append(loss)
#             offset += n_labels

#         return torch.stack(sample_losses).mean()
        

# def main():
#     set_seed(42)

#     use_fp16 = "fp16" if torch.cuda.is_available() else "no"
#     ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)

#     accelerator = Accelerator(mixed_precision=use_fp16, dataloader_config=dataloader_config)
#     special_tokens = SpecialTokens()

#     model_name = "FacebookAI/roberta-base"

#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     base_model = AutoModel.from_pretrained(model_name,add_pooling_layer=False)

#     num_added_tokens = tokenizer.add_special_tokens(
#         {"additional_special_tokens": special_tokens.SPECIAL_TOKENS}
#     )
    

#     if num_added_tokens > 0:
#         base_model.resize_token_embeddings(len(tokenizer))

#     label_token_id = tokenizer.convert_tokens_to_ids(special_tokens.L_TOKEN)
#     model = GLiNERTextClassifier(
#         base_model=base_model, label_token_id=label_token_id
#     )
#     if accelerator.is_main_process:
#         print(f"Added {num_added_tokens} new tokens to tokenizer.")

#     train_data = ParquetIterableClassificationDataset(
#         "/kaggle/working/multilabel_dataset_train.parquet",
#         tokenizer=tokenizer,
#         special_tokens=special_tokens,
#     )

#     train_loader = DataLoader(
#         train_data,
#         batch_size=48,
#         collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
#     )

#     val_data = ParquetIterableClassificationDataset(
#         "/kaggle/working/multilabel_dataset_val.parquet",
#         tokenizer=tokenizer,
#         special_tokens=special_tokens,
#         max_negatives=None,
#     )

#     val_loader = DataLoader(
#         val_data,
#         batch_size=64,
#         collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id),
#     )

#     optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
#     criterion = HybridClassificationLoss()

#     model, optimizer, train_loader, val_loader = accelerator.prepare(
#         model, optimizer, train_loader, val_loader
#     )
#     # Calculate total batches assigned to the local process
#     num_epochs = 2
#     total_rows = len(train_data)
#     per_process_rows = math.ceil(total_rows / accelerator.num_processes)
#     total_train_batches = math.ceil(per_process_rows / train_loader.batch_size)

#     for epoch in range(num_epochs):
#         model.train()
#         total_train_loss = 0.0
#         progress_bar = tqdm(
#         train_loader,
#         total=total_train_batches,
#         desc=f"Epoch {epoch + 1}/{num_epochs}",
#         disable=not accelerator.is_local_main_process)
            

#         for step, batch in enumerate(progress_bar):
#             optimizer.zero_grad()

#             logits = model(
#                 input_ids=batch["input_ids"],
#                 attention_mask=batch["attention_mask"],
#             )
#             targets = batch["targets"]

#             # Direct scalar 1D Loss computation
#             loss = criterion(logits=logits,targets=batch["targets"],num_labels_per_sample=batch["num_labels_per_sample"],is_multilabel=batch["is_multilabel"])
#             # loss = criterion(logits, targets)

#             accelerator.backward(loss)
#             accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             optimizer.step()

#             total_train_loss += loss.item()
#             current_avg_loss = total_train_loss / (step + 1)
#             progress_bar.set_postfix({"loss": f"{current_avg_loss:.4f}"})
#         val_metrics = evaluate(
#             model=model,
#             val_dataloader=val_loader,
#             accelerator=accelerator,
#             special_tokens=special_tokens,
#             tokenizer=tokenizer,
#             threshold=0.5,
#             criterion=criterion,
#             num_samples_to_print=20,
#         )
#         accelerator.print(f"Epoch {epoch + 1} Validation Metrics: {val_metrics}")

#     accelerator.wait_for_everyone()
#     unwrapped_model = accelerator.unwrap_model(model)
#     if accelerator.is_main_process:
#         torch.save(unwrapped_model.state_dict(), "gliner_classifier.pt")
#         tokenizer.save_pretrained("./tokenizer")
#         accelerator.print("Model successfully saved!")


# if __name__ == "__main__":
#     main()

In [8]:
# ! accelerate launch --num_processes 2 ../working/train_script.py

In [9]:
# kwargs_handlers=[ddp_kwargs],